# Standard 3B DPO baseline: Stage 1 and Stage 2

This is the clean diagnostic baseline requested:

- Existing full-data 3B SFT checkpoint
- Original full gold-supported curriculum pairs
- Stage 1: C3 > C1
- Stage 2: C3 > C2
- Ordinary DPO for both stages
- Identical training hyperparameters for both stages
- No DPO-D ties, APO, Cal-DPO, adaptive NLL, MAP filtering, retention anchor, or L2-SP
- Stage 2 automatically starts from the Stage 1 checkpoint selected by dev MRR

The run uses a new output directory and does not overwrite previous experiments.


In [ ]:
from pathlib import Path
import importlib
import json
import os
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

PROJECT_ROOT = Path("/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cse_dpo.train_factoid_three_stage_dpo as trainer
trainer = importlib.reload(trainer)

Config = trainer.Config
load_stage_rows = trainer.load_stage_rows
run_three_stage = trainer.run_three_stage

FULL_DATA_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/class_judgments/candidate_banks_train_full_fresh_c3_c1_v1_gold_c3_gold_supported/merged/staged_curriculum_pairs"
SFT_3B = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_3b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr"
OUTPUT_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/three_stage_dpo_qwen25_3b_standard_dpo_two_stage_gold_supported_full_bp2048"

STANDARD_STAGE = {
    "objective": "dpo",
    "beta": 0.05,
    "learning_rate": 5e-7,
    "epochs": 8,
    "batch_size": 1,
    "gradient_accumulation": 8,
    "max_length": 4096,
    "train_backprop_max_length": 2048,
    "optimizer": "AdamW",
    "weight_decay": 0.01,
    "warmup_fraction": 0.05,
    "max_grad_norm": 1.0,
    "lora_dropout": 0.05,
    "eval_every_updates": 10,
    "early_stopping_patience": 8,
    "selection_metric": "dev_mrr",
    "semantic_judge_enabled": False,
    "retention_anchor_weight": 0.0,
    "l2sp_weight": 0.0,
}

CFG = Config(
    model_preset="qwen25_3b",
    dataset_preset="gold_supported_full_merged",
    staged_root=str(FULL_DATA_ROOT),
    initial_adapter=str(SFT_3B),
    output_root=str(OUTPUT_ROOT),
    smoke_test=False,
    stop_after_stage="format_alignment",
    skip_stages=[],
    objective="dpo",
    beta=0.05,
    learning_rate=5e-7,
    optimizer="AdamW",
    weight_decay=0.01,
    warmup_fraction=0.05,
    lora_dropout=0.05,
    batch_size=1,
    gradient_accumulation=8,
    max_length=4096,
    train_backprop_max_length=2048,
    generated_eval_max_seq_length=4096,
    generated_eval_max_new_tokens=64,
    evaluate_generated_dev=True,
    semantic_judge_enabled=False,
    stage_settings={
        "concept_learning": dict(STANDARD_STAGE),
        "format_alignment": dict(STANDARD_STAGE),
    },
)

print(json.dumps({
    "model": CFG.model_preset,
    "base_model": CFG.base_model,
    "initial_sft_adapter": CFG.initial_adapter,
    "dataset": CFG.staged_root,
    "output_root": CFG.output_root,
    "stop_after_stage": CFG.stop_after_stage,
    "stage_settings": CFG.stage_settings,
}, indent=2))


In [ ]:
assert SFT_3B.exists(), SFT_3B
assert FULL_DATA_ROOT.exists(), FULL_DATA_ROOT

stages = load_stage_rows(CFG)
assert len(stages["concept_learning"]) == 293
assert len(stages["format_alignment"]) == 1319
assert CFG.stage_settings["concept_learning"] == CFG.stage_settings["format_alignment"]

print(json.dumps({
    "output_root_already_exists": OUTPUT_ROOT.exists(),
    "stage_1_pairs_before_length_filters": len(stages["concept_learning"]),
    "stage_1_questions": len({row["question_id"] for row in stages["concept_learning"]}),
    "stage_2_pairs_before_length_filters": len(stages["format_alignment"]),
    "stage_2_questions": len({row["question_id"] for row in stages["format_alignment"]}),
    "stage_2_initialization": "best Stage 1 adapter selected by dev_mrr",
    "training_context": 2048,
    "evaluation_context": 4096,
}, indent=2))


In [ ]:
# Start the two-stage run. Completed stages are reused automatically on rerun.
results = run_three_stage(CFG)
print(json.dumps(results, indent=2, default=str))
